In [1]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import JSONLoader

# Step 1: Load your JSONL into documents
import json
import os 
OUTPUT_DIR = "../output"

dataset_location = os.path.join(OUTPUT_DIR, "math_dataset_corr_ai.jsonl")

docs = []
with open(dataset_location, "r") as f:
    for line in f:
        item = json.loads(line)
        docs.append(item["question"] + "\n" + item["raw_correction"])  # You can combine fields

# Step 2: Embed documents
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(docs, embeddings)

# Save the index
vectorstore.save_local("math_index")


ModuleNotFoundError: Module langchain_community.embeddings not found. Please install langchain-community to access this module. You can install it using `pip install -U langchain-community`

In [ ]:
retrieved_docs = vectorstore.similarity_search("Comment résoudre 7 × 8 + 13 ?", k=3)

context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""Voici des explications utiles:\n{context}\n\nMaintenant, réponde à la question suivante de façon claire:\nComment résoudre 7 × 8 + 13 ?"""
response = openai.ChatCompletion.create(
    model="gpt-4",
    messages=[{"role": "user", "content": prompt}]
)


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = "Solve this step by step: 5 + 11 * 4 + 3 / 3 - 7 = ?"
response = pipe(prompt, max_new_tokens=200)
print(response[0]["generated_text"])


tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

c:\Users\night\Documents\111Project\Math_College_Dataset\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\night\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu


: 